# Laboratório 6 — Depth Map (Mapa de Profundidade)

**Disciplina:** ESZA019 — Visão Computacional — UFABC 2026.2
**Professor:** Celso Setsuo Kurashima
**Equipe:** Ctrl+C, Ctrl+V e Fé

### Integrantes da equipe
- Lucas Rodrigues Teixeira — RA 11202131394
- Pedro Henrique Garcez Silva — RA 11202130642
- Roberto Sene Azevedo — RA 11202020360

**Data de realização dos experimentos:** 15 a 20/07/2026
**Data de publicação do relatório:** 20/07/2026


## Sumário

1. [Introdução](#1-introdução)
2. [Fundamentação teórica](#2-fundamentação-teórica)
   - 2.1 [Disparidade e profundidade](#21-disparidade-e-profundidade)
   - 2.2 [Retificação estéreo](#22-retificação-estéreo)
   - 2.3 [Block Matching para correspondência densa](#23-block-matching-para-correspondência-densa)
   - 2.4 [Diagrama de blocos do processo completo](#24-diagrama-de-blocos-do-processo-completo)
3. [Ambiente e instruções de reprodução](#3-ambiente-e-instruções-de-reprodução)
4. [Procedimentos experimentais (Parte 2)](#4-procedimentos-experimentais-parte-2)
   - 4.1 [(i) Calibração estéreo](#41-i-calibração-estéreo)
   - 4.2 [Validação dos algoritmos em imagens gravadas](#42-validação-dos-algoritmos-em-imagens-gravadas)
   - 4.3 [(ii) Block Matching: sintonia dos parâmetros](#43-ii-block-matching-sintonia-dos-parâmetros)
   - 4.4 [(iii) Do mapa de disparidade ao mapa de profundidade](#44-iii-do-mapa-de-disparidade-ao-mapa-de-profundidade)
   - 4.5 [(iv e v) Medidas de distância e erros](#45-iv-e-v-medidas-de-distância-e-erros)
   - 4.6 [(vi) Programa completo de medição de distância](#46-vi-programa-completo-de-medição-de-distância)
5. [Análise e discussão](#5-análise-e-discussão)
6. [Conclusões](#6-conclusões)
7. [Declaração de uso de Inteligência Artificial Generativa](#7-declaração-de-uso-de-inteligência-artificial-generativa)
8. [Referências](#8-referências)


## 1. Introdução

No Laboratório 5 a equipe construiu e calibrou uma **câmera estereoscópica** de baixo custo, obtendo os
parâmetros que permitem **retificar** o par de imagens. Neste Laboratório 6 damos o passo seguinte e
final da cadeia estéreo: transformar as duas vistas em **medida de distância**.

O caminho passa por dois mapas. Primeiro o **mapa de disparidade**, que registra, para cada pixel, o
deslocamento horizontal entre a imagem esquerda e a direita. Depois o **mapa de profundidade**, obtido da
disparidade pela relação inversa $Z = f\,B/d$. Com ele é possível **medir a que distância está um objeto**
— capacidade que sustenta aplicações como navegação de robôs, desvio de obstáculos e, no caso do Trabalho
Final da equipe (detecção de uso de EPI), a delimitação da zona monitorada.

Este relatório descreve a fundamentação teórica, o diagrama de blocos do processo completo, os
procedimentos executados (sintonia do *Block Matching*, calibração disparidade→profundidade e medições) e
a **análise crítica dos resultados**, incluindo a influência dos parâmetros do algoritmo sobre a
qualidade das medidas.


## 2. Fundamentação teórica

### 2.1 Disparidade e profundidade

Em um par estéreo **retificado**, um mesmo ponto do mundo aparece na **mesma linha** nas duas imagens,
deslocado apenas horizontalmente. Esse deslocamento é a **disparidade**:

$$d = x_L - x_R$$

A relação com a profundidade $Z$ decorre da semelhança de triângulos no modelo *pinhole*:

$$Z = \frac{f \cdot B}{d}$$

onde $f$ é a distância focal (em pixels, na imagem retificada) e $B$ é a *baseline* (distância entre os
centros ópticos). A relação é **inversa e não linear**: disparidade grande ⇒ objeto **próximo**;
disparidade pequena ⇒ objeto **distante**; disparidade nula ⇒ ponto no infinito.

Duas consequências práticas importantes:

1. **Resolução em profundidade cai com a distância.** Como $Z \propto 1/d$, um erro de 1 px na disparidade
   causa erro pequeno perto e **muito grande** longe.
2. **A faixa de disparidade define a faixa de medição.** Se o algoritmo só procura disparidades no
   intervalo $[d_{\min}, d_{\min}+N)$, então só consegue medir profundidades entre
   $f B/(d_{\min}+N-1)$ e $f B/d_{\min}$. Este ponto será central na análise deste laboratório.


### 2.2 Retificação estéreo

A busca por correspondências só é barata porque as imagens são **retificadas**: as linhas epipolares
tornam-se **horizontais e alinhadas**, reduzindo a busca de um problema 2D para **1D** ao longo da linha.
A retificação é feita com os mapas gerados na calibração estéreo do Lab 5
(`cv2.stereoRectify` + `cv2.initUndistortRectifyMap`), aplicados a cada quadro com `cv2.remap`. Sem essa
etapa, o *Block Matching* falha, pois o ponto correspondente não estaria na mesma linha.


### 2.3 Block Matching para correspondência densa

O **Block Matching (BM)** estima a disparidade comparando **blocos** (janelas quadradas) de pixels: para
cada bloco da imagem esquerda, desliza-se o bloco ao longo da linha correspondente na imagem direita,
buscando o deslocamento que **minimiza a diferença** (tipicamente SAD — soma das diferenças absolutas).
No OpenCV isso é `cv2.StereoBM`; a variante `cv2.StereoSGBM` (*Semi-Global*) agrega custos em várias
direções e produz mapas mais suaves, ao custo de mais processamento.

Os parâmetros mais relevantes:

| Parâmetro | Efeito |
|-----------|--------|
| `numDisparities` | Tamanho da **faixa de busca** de disparidade (múltiplo de 16). Define a **distância mínima** mensurável. |
| `minDisparity` | Disparidade inicial da busca. Define a **distância máxima** mensurável. |
| `blockSize` | Tamanho da janela. Pequeno = mais detalhe e mais ruído; grande = mais suave e menos detalhe. |
| `uniquenessRatio` | Margem exigida do melhor casamento sobre o segundo. Alto = descarta mais pontos. |
| `textureThreshold` | Textura mínima do bloco. Regiões lisas são descartadas. |
| `speckleWindowSize` / `speckleRange` | Filtram manchas isoladas ("speckles") no mapa. |

> **Observação importante:** o `StereoBM` retorna a disparidade em **ponto fixo**, multiplicada por 16.
> Por isso todos os programas dividem o resultado por 16 antes de usá-lo.


### 2.4 Diagrama de blocos do processo completo

Conforme solicitado na Parte 1 do roteiro, o diagrama abaixo ilustra o processo desde a construção da
câmera estéreo até a obtenção do mapa de profundidade e da medida de distância.

```
  [ LAB 5 ]                                    [ LAB 6 ]
  ---------                                    ---------

+---------------------+
| (1) Construcao da   |   duas webcams iguais, eixos opticos paralelos,
|     camera estereo  |   baseline ~ distancia interpupilar, fixacao rigida
+----------+----------+
           |
           v
+---------------------+
| (2) Captura do      |   capture_images_abc.py -> 15 pares L/R do tabuleiro
|     padrao (xadrez) |
+----------+----------+
           |
           v
+---------------------+
| (3) Calibracao      |   calibrate_abc.py: calibracao mono (KL,KR,distL,distR)
|     estereo         |   + stereoCalibrate (R, T, E, F) + stereoRectify (Q)
+----------+----------+
           |
           v
   [ params_py.xml ]  <-- mapas de retificacao, K, dist, R, T, E, F, Q, baseline
           |
===========|==================================================================
           v
+---------------------+
| (4) Retificacao     |   cv2.remap: linhas epipolares horizontais alinhadas
|     dos quadros     |
+----------+----------+
           |
           v
+---------------------+
| (5) Block Matching  |   disparity_params_gui.py: sintonia dos parametros
|  -> MAPA DE         |   (numDisparities, blockSize, filtros)
|     DISPARIDADE     |
+----------+----------+
           |
           v
 [ depth_estimation_params_py.xml ]  <-- parametros do BM
           |
           v
+---------------------+
| (6) Calibracao      |   disparity2depth_calib.py: amostras (d, Z_real)
|  disparidade ->     |   ajuste por minimos quadrados  Z = M*(1/d) + C
|  PROFUNDIDADE       |   -> grafico Profundidade vs Disparidade
+----------+----------+
           |
           v
 [ M , C salvos no xml ]
           |
           v
+---------------------------------------------+
| (7) MAPA DE PROFUNDIDADE e MEDIDA DE DISTANCIA|
|   obstacle_avoidance.py  (obstaculo proximo)  |
|   medir_distancia_objeto.py (objeto por clique)|
+---------------------------------------------+
```


## 3. Ambiente e instruções de reprodução

Ambiente **Linux**, **Python 3**, **OpenCV 4.x**, **NumPy** e **Matplotlib**. A câmera estéreo é a
construída no Lab 5, mantida **fixa** (qualquer movimento relativo entre as webcams invalida a
calibração). Um guia detalhado está em [`PASSO_A_PASSO.md`](PASSO_A_PASSO.md); em resumo:

```bash
# 0) O params_py.xml do Lab 5 deve estar nesta pasta
cp "../Laboratório 5/data/params_py.xml" .

# 1) (opcional) validar os algoritmos sem webcam, em imagens gravadas
python3 disparidade_offline.py

# 2) sintonizar o Block Matching -> depth_estimation_params_py.xml   ('s' salva, 'q' sai)
python3 disparity_params_gui.py

# 3) calibrar disparidade -> profundidade (M e C) + grafico Z vs d   ('q' encerra)
python3 disparity2depth_calib.py

# 4) medir distancia de obstaculos / de um objeto especifico
python3 obstacle_avoidance.py
python3 medir_distancia_objeto.py
```

> **Índices das câmeras.** No Linux ajuste `CamL_id`/`CamR_id` conforme `v4l2-ctl --list-devices`. Se o
> mapa de disparidade sair **totalmente preto**, a causa mais comum é a **inversão** entre câmera
> esquerda e direita (disparidades negativas são descartadas).


## 4. Procedimentos experimentais (Parte 2)

### 4.1 (i) Calibração estéreo

Conforme o roteiro, foi reaproveitada a calibração estéreo realizada no Laboratório 5 (mesma câmera,
mantida fixa), cujos parâmetros estão em `params_py.xml`. Os valores relevantes para este laboratório
são a **distância focal da imagem retificada** e a ***baseline***, que definem a constante da relação
$Z = f B/d$. A célula abaixo relê o arquivo e calcula essa constante.


In [ ]:
import cv2
import numpy as np

fs = cv2.FileStorage("params_py.xml", cv2.FILE_STORAGE_READ)
Q = fs.getNode("Q").mat()                       # matriz de reprojecao (Lab 5)
baseline_mm = fs.getNode("baseline_mm").real()  # distancia entre as cameras
fs.release()

f_px = Q[2, 3]              # distancia focal da imagem retificada, em pixels
B_cm = baseline_mm / 10.0   # baseline em cm

print(f"f  = {f_px:.2f} px")
print(f"B  = {baseline_mm:.2f} mm = {B_cm:.3f} cm")
print(f"Relacao teorica:  Z(cm) = f*B/d = {f_px*B_cm:.1f} / d")


**Resultado.** $f = 676{,}57$ px e $B = 67{,}29$ mm $= 6{,}729$ cm, de onde

$$Z_{\text{teórico}}(\text{cm}) = \frac{f \cdot B}{d} = \frac{4552{,}9}{d}.$$

Essa expressão será a referência para avaliar, na análise, a calibração empírica obtida no item (iii).


### 4.2 Validação dos algoritmos em imagens gravadas

Antes de usar as webcams, os algoritmos de correspondência foram validados em **pares de imagens já
gravados** (programa `disparidade_offline.py`), o que permite verificar o funcionamento sem depender do
arranjo físico. Foram usados o par estéreo de alta resolução (`im0/im1`) com **StereoSGBM** e o clássico
par *tsukuba* com **StereoBM**.

![Mapa de disparidade obtido com StereoSGBM no par im0/im1](demo_disparidade_sgbm.png)

*Figura 1 — Mapa de disparidade (StereoSGBM) do par `im0/im1`. Cores quentes = disparidade alta =
objeto **próximo** (a motocicleta); cores frias = disparidade baixa = fundo **distante**.*

![Mapa de disparidade obtido com StereoBM no par tsukuba](demo_disparidade_bm_tsukuba.png)

*Figura 2 — Mapa de disparidade (StereoBM) do par `tsukuba`, reproduzindo o exemplo do tutorial do
OpenCV.*

A Figura 1 mostra claramente o comportamento esperado: o objeto em primeiro plano destaca-se do fundo, e
o gradiente do piso reflete o aumento contínuo de distância. Isso confirma que o *pipeline* de
correspondência está correto e que eventuais problemas nas etapas seguintes vêm da **sintonia dos
parâmetros** ou do arranjo físico, não do algoritmo.


### 4.3 (ii) Block Matching: sintonia dos parâmetros

Com a câmera estéreo, executou-se o `disparity_params_gui.py`, que retifica os quadros das duas webcams
com os mapas de `params_py.xml` e calcula a disparidade com `cv2.StereoBM`, permitindo ajustar os
parâmetros por barras deslizantes e salvá-los com a tecla `s`.

**Dificuldade encontrada.** Durante a sintonia, o mapa de disparidade chegou a ficar **completamente
preto**. Isso não significa um mapa "limpo", e sim que **todos os pixels foram invalidados** — efeito de
filtros excessivamente agressivos (`uniquenessRatio`, `textureThreshold`, `speckleWindowSize`) e/ou de
`minDisparity` alto demais. Reduzindo os filtros o relevo da cena voltou a aparecer.

Os parâmetros efetivamente salvos em `depth_estimation_params_py.xml` foram:


In [ ]:
fs = cv2.FileStorage("depth_estimation_params_py.xml", cv2.FILE_STORAGE_READ)
numDisparities = int(fs.getNode("numDisparities").real())
blockSize = int(fs.getNode("blockSize").real())
M = fs.getNode("M").real()
C = fs.getNode("C").real()
fs.release()

print(f"numDisparities = {numDisparities}   (o programa usa max(16, valor) -> efetivo {max(16, numDisparities)})")
print(f"blockSize      = {blockSize}")
print(f"M = {M:.2f}   C = {C:.2f}      ->   Z(cm) = {M:.1f}*(1/d) + ({C:.1f})")


| Parâmetro salvo | Valor | Observação |
|-----------------|-------|------------|
| `numDisparities` | 0 (efetivo **16**) | valor mínimo da barra; o programa força o mínimo de 16 |
| `blockSize` | 5 | valor mínimo da barra |
| `minDisparity` | não salvo (usa padrão **5**) | define a distância máxima mensurável |

Como se verá na análise, essa combinação — a **mais restritiva possível** — limitou fortemente a faixa de
distâncias que o sistema conseguiu medir.


### 4.4 (iii) Do mapa de disparidade ao mapa de profundidade

Com os parâmetros do BM definidos, executou-se o `disparity2depth_calib.py`. O procedimento consiste em
posicionar um objeto a **distâncias reais conhecidas** (medidas com trena), **clicar** sobre ele no mapa
de disparidade para amostrar $d$ e informar a distância real $Z$. Com as amostras, o programa ajusta por
**mínimos quadrados** o modelo

$$Z = M\cdot\frac{1}{d} + C$$

Foram coletadas **5 amostras**, cobrindo de ~22 cm a ~222 cm. O ajuste resultou em

$$\boxed{Z(\text{cm}) = \frac{1019{,}53}{d} - 53{,}45}$$

e os coeficientes $M$ e $C$ foram gravados em `depth_estimation_params_py.xml` para uso nos programas de
medição. O gráfico exigido pelo roteiro é apresentado abaixo.

![Gráfico Profundidade vs Disparidade com as amostras e a curva ajustada](profundidade_vs_disparidade.png)

*Figura 3 — Profundidade × Disparidade: pontos vermelhos são as amostras medidas pela equipe e a curva
azul é o modelo ajustado $Z = 1019{,}5/d - 53{,}4$.*

Nota-se que os pontos apresentam **dispersão considerável** em torno da curva — em particular, amostras
com disparidades muito próximas (≈5,2 e ≈5,6 px) correspondem a distâncias reais muito diferentes (46 cm
e 222 cm). A causa dessa inconsistência é analisada na Seção 5.


### 4.5 (iv e v) Medidas de distância e erros

Com o modelo calibrado, executou-se o `obstacle_avoidance.py`, que converte a disparidade em
profundidade, identifica a **região mais próxima** da cena e exibe sua distância, emitindo alerta quando
o obstáculo está muito perto.

![Detecção de obstáculo próximo com a distância estimada](deteccao_de_obstaculo_experimento.jpeg)

*Figura 4 — Execução do `obstacle_avoidance.py`: a mão em primeiro plano é identificada como o obstáculo
mais próximo, com distância estimada de **15 cm** e o alerta "OBSTACULO PROXIMO!".*

As medições registradas nos experimentos (janelas e console) foram:

| # | Objeto | Distância medida (cm) | Distância real (cm) | Erro (cm) | Erro (%) |
|---|--------|-----------------------|---------------------|-----------|----------|
| 1 | Mão fechada (obstáculo, Figura 4) | 15 | _a medir_ | | |
| 2 | Mão aberta (Figura 5) | 55 | _a medir_ | | |
| 3 | Objeto próximo (console) | 28 | _a medir_ | | |
| 4 | Objeto afastado (console) | 80 | _a medir_ | | |

> **Nota.** Os valores da coluna "distância medida" são os efetivamente exibidos pelos programas e estão
> documentados nas Figuras 4 e 5. As **distâncias reais** correspondentes (com trena) não ficaram
> registradas nos arquivos do experimento; para fechar a tabela de erros exigida pelo item (v), basta
> repetir as medições anotando a distância real de cada objeto. A Seção 5 analisa, com base nos dados
> disponíveis, a **ordem de grandeza do erro** e sua causa.


### 4.6 (vi) Programa completo de medição de distância

Por fim, o `medir_distancia_objeto.py` implementa o programa completo pedido no item (vi): ele retifica o
par, calcula a disparidade, converte em profundidade pelo modelo calibrado e, ao **clique do mouse**,
informa a distância do objeto apontado. O programa foi ajustado ao tema do **Trabalho Final da equipe
(Detecção de Uso de EPI)**: além da distância, ele indica se o objeto está dentro da **zona monitorada**
(1,5 a 3 m), critério que no trabalho final servirá para avaliar o EPI apenas de quem está na região de
acesso.

![Medição de distância de um objeto por clique](medicao_de_distancia_experimento.jpeg)

*Figura 5 — Execução do `medir_distancia_objeto.py`: ao clicar sobre a mão, o programa informa
"Distancia: 55 cm (fora da zona)" — corretamente sinalizando que o objeto está **fora** da faixa de
1,5–3 m definida para o Trabalho Final.*

Trecho central do programa (conversão disparidade → distância no ponto clicado):


In [ ]:
def ao_clicar(event, x, y, flags, param):
    """Ao clicar, mede a disparidade media numa janela 6x6 e converte em distancia."""
    if event == cv2.EVENT_LBUTTONDOWN and estado["disp"] is not None:
        jan = estado["disp"][max(0, y-3):y+3, max(0, x-3):x+3]
        d = np.mean(jan[jan > 0]) if np.any(jan > 0) else 0.0   # disparidade media
        if d <= 0:
            estado["texto"] = "sem disparidade nesse ponto"; return
        z = M * (1.0 / d) + C                                    # modelo calibrado
        dentro = ZONA_MIN_CM <= z <= ZONA_MAX_CM                 # zona monitorada (tema EPI)
        estado["texto"] = f"Distancia: {z:.0f} cm " + ("(na zona)" if dentro else "(fora da zona)")


## 5. Análise e discussão

### 5.1 Influência dos parâmetros: a faixa de busca de disparidade

A análise mais relevante deste laboratório diz respeito ao efeito de `numDisparities` e `minDisparity`
sobre a **faixa de distâncias mensurável**. Com $d_{\min} = 5$ e $N = 16$ (os valores efetivamente
usados), a busca cobre apenas $d \in [5,\ 21)$ px. Aplicando $Z = 4552{,}9/d$:

$$Z_{\max} = \frac{4552{,}9}{5} \approx 911\ \text{cm}, \qquad Z_{\min} = \frac{4552{,}9}{20} \approx 228\ \text{cm}$$

Ou seja, **a configuração usada só consegue medir objetos a mais de ~2,3 m**. Comparando com as
distâncias reais empregadas na calibração:

| Distância real $Z$ | Disparidade teórica $d = 4552{,}9/Z$ | Dentro da faixa $[5, 21)$? |
|--------------------|--------------------------------------|-----------------------------|
| 22 cm | 206,9 px | Não |
| 46 cm | 99,0 px | Não |
| 85 cm | 53,6 px | Não |
| 143 cm | 31,8 px | Não |
| 222 cm | 20,5 px | Sim (no limite) |

Apenas a amostra mais distante caiu dentro da faixa de busca. Para todas as demais, a disparidade
verdadeira estava **muito além** do que o algoritmo procurava; o *Block Matching* então convergiu para
casamentos espúrios de baixa disparidade (5–10 px), o que explica diretamente a **dispersão** observada na
Figura 3 e o fato de amostras quase idênticas em $d$ corresponderem a distâncias reais muito diferentes.

### 5.2 Consistência entre a calibração empírica e o modelo físico

O modelo físico prevê $Z = (f B)/d$ com $f B = 4552{,}9$ cm·px, enquanto o ajuste empírico resultou em
$M = 1019{,}5$ cm·px — uma diferença de **~4,5×**, com um termo constante $C = -53{,}4$ cm que não tem
significado físico direto (deveria ser ≈ 0) e que, no ajuste, funcionou como compensação do viés
sistemático. Essa discrepância é **coerente com o diagnóstico da Seção 5.1**: como as disparidades
medidas não correspondiam às reais, os coeficientes ajustados não podiam reproduzir a física do arranjo.

Vale notar que a calibração estéreo em si (Lab 5) está **correta** — erro de reprojeção de 0,13 px (mono)
e 1,00 px (estéreo), *baseline* de 67,3 mm compatível com a distância interpupilar. O problema está
exclusivamente na **sintonia do Block Matching**, não na geometria da câmera.

### 5.3 Correção recomendada

Para medir a partir de ~35 cm, a faixa de busca precisa cobrir $d = 4552{,}9/35 \approx 130$ px, o que
exige `numDisparities` $\geq 144$ (múltiplo de 16). Recomenda-se repetir os itens (ii) a (v) com:

- `numDisparities` = **144 a 224** (em vez de 16);
- `minDisparity` = **0** (em vez de 5), liberando as distâncias maiores;
- `blockSize` = **9 a 15** (em vez de 5), reduzindo o ruído sem perder muito detalhe;
- filtros (`uniquenessRatio`, `speckleWindowSize`) elevados apenas até o ruído sumir, **sem** apagar o
  objeto — o mapa deve continuar mostrando o relevo da cena.

Com esses ajustes, espera-se que $M$ convirja para próximo de $f B \approx 4553$ e que $C \to 0$,
tornando as medidas fisicamente consistentes.

### 5.4 Demais fatores de influência

Além da faixa de busca, observou-se que a qualidade do mapa depende de: **textura** da cena (superfícies
lisas, como paredes brancas, não geram disparidade — visível nas regiões vazias das Figuras 4 e 5);
**iluminação** e diferença de exposição entre as duas webcams; e o **compromisso do `blockSize`** entre
detalhe e ruído. Também vale lembrar a característica intrínseca do método: como $Z \propto 1/d$, a
**incerteza cresce com o quadrado da distância**, de modo que a mesma imprecisão de 1 px na disparidade
produz erro muito maior em objetos distantes.


## 6. Conclusões

O Laboratório 6 cumpriu seu objetivo de percorrer toda a cadeia estéreo, da **retificação** ao **mapa de
disparidade**, deste ao **mapa de profundidade** e, finalmente, à **medida de distância** de objetos
reais. Os programas desenvolvidos — sintonia do *Block Matching*, calibração disparidade→profundidade,
detecção de obstáculo e medição por clique — funcionaram de ponta a ponta, produzindo o gráfico
Profundidade × Disparidade e medições em tempo real (Figuras 3 a 5).

A validação em imagens gravadas (Figura 1) confirmou a correção do *pipeline* de correspondência, e a
calibração estéreo herdada do Lab 5 mostrou-se precisa (erro de reprojeção de 0,13 px mono e 1,00 px
estéreo). O principal aprendizado, porém, veio da **análise crítica dos parâmetros**: a faixa de busca
utilizada (`numDisparities` efetivo de 16 com `minDisparity` 5) restringia a medição a distâncias
superiores a ~2,3 m, enquanto os objetos foram posicionados entre 15 cm e 222 cm. Essa incompatibilidade
explica quantitativamente tanto a dispersão das amostras quanto o desvio de ~4,5× entre a constante
empírica ($M = 1019{,}5$) e a prevista pela física do arranjo ($fB = 4552{,}9$).

Fica assim evidenciada a lição central do experimento: em visão estéreo, **a geometria bem calibrada não
basta** — os parâmetros do algoritmo de correspondência precisam ser escolhidos de forma coerente com a
**faixa de distâncias de interesse**. A correção é direta (ampliar `numDisparities` para 144–224, zerar
`minDisparity` e aumentar o `blockSize`) e deve tornar as medidas fisicamente consistentes. Esse
resultado é diretamente aproveitável no Trabalho Final da equipe, em que a medição de distância delimita
a zona de monitoramento do uso de EPI.


## 7. Declaração de uso de Inteligência Artificial Generativa

Em atendimento à **Portaria CNPq nº 2664/2026**, a equipe declara ter utilizado um assistente de IA
generativa (Anthropic — Claude) apenas como apoio à **formatação e organização textual** deste
documento — estruturação das seções, revisão de escrita e formatação do notebook. Todo o conteúdo
técnico, os dados e os resultados são de autoria da equipe, que é integralmente responsável pelo
material final.


## 8. Referências

1. LearnOpenCV — *Stereo Camera Depth Estimation With OpenCV (Python/C++)*. Disponível em:
   https://learnopencv.com/depth-perception-using-stereo-camera-python-c/
   (código: https://github.com/spmallick/learnopencv/tree/master/Depth-Perception-Using-Stereo-Camera)
2. OpenCV — *Depth Map from Stereo Images*. Disponível em:
   https://docs.opencv.org/4.x/dd/d53/tutorial_py_depthmap.html
3. LearnOpenCV — *Introduction to Epipolar Geometry and Stereo Vision*. Disponível em:
   https://learnopencv.com/introduction-to-epipolar-geometry-and-stereo-vision/
4. LearnOpenCV — *Making A Low-Cost Stereo Camera Using OpenCV*. Disponível em:
   https://learnopencv.com/making-a-low-cost-stereo-camera-using-opencv/
5. LOOP, C.; ZHANG, Z. *Computing Rectifying Homographies for Stereo Vision*. IEEE CVPR, 1999.
6. LearnOpenCV — *Geometry of Image Formation*. Disponível em:
   https://learnopencv.com/geometry-of-image-formation/
7. TRUCCO, E.; VERRI, A. *Introductory Techniques for 3-D Computer Vision*. Prentice Hall, 1998.
